In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report,accuracy_score
from joblib import dump, load

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy.stats import mode
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import Binarizer
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder

from sklearn.datasets import fetch_openml

from xai_aux import CFCalculations
from xai_aux import simulate_noise_effect

In [2]:
warnings.filterwarnings("ignore", message="X has feature names, but RandomForestClassifier was fitted without feature names")
warnings.filterwarnings("ignore", message="X has feature names, but LogisticRegression was fitted without feature names")
warnings.filterwarnings("ignore", message="X does not have valid feature names, but OneHotEncoder was fitted with feature names")
warnings.filterwarnings("ignore", message="X does not have valid feature names, but StandardScaler was fitted with feature names")
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", message=".*At this time, the v2.11\\+ optimizer.*")

In [3]:
credit = pd.read_csv('data/gsc/cs-training.csv').dropna() # very big dataset anyway
credit = credit.sample(frac=0.4)
credit.drop(columns=['Unnamed: 0'],inplace=True)
df = credit.drop(columns=["SeriousDlqin2yrs"])  # Features
y = credit["SeriousDlqin2yrs"].values  # Target variable
cat_feat = []
num_feat = df.columns.to_list()
print(len(df))

48108


In [11]:
results, datasets = simulate_noise_effect(df, num_feat, cat_feat, y, max_noise=5., step=0.5, min_frequency=None)
noise_levels, accuracies = zip(*results)
plt.plot(noise_levels, accuracies, marker='o')
plt.xlabel('Noise Level')
plt.ylabel('Classifier Accuracy')
plt.title('Effect of Noise on Classifier Accuracy')
plt.grid()
plt.show()

KeyError: 'RevolvingUtilizationOfUnsecuredLines'

In [5]:
n_features = len(df.columns)

In [6]:
def get_polytopes(enc):
    # Get transformed feature names
    feature_names = enc.get_feature_names_out()
    #print("Encoded Feature Names:", feature_names)  # Debugging step

    # Extract number of features per original category
    feature_counts = []
    current_feature = None
    count = 0

    for feature in feature_names:
        original_category = feature.split("_")[0]  # Extract the original category index
        if original_category != current_feature:
            if count > 0:  # Append count for previous category
                feature_counts.append(count)
            current_feature = original_category
            count = 1  # Start counting for new category
        else:
            count += 1
    feature_counts.append(count)  # Append the last count

    # Generate list of lists with sequential numbering
    start = 1
    polytopes = []

    for count in feature_counts:
        polytopes.append(list(range(start, start + count)))
        start += count
    polytopes = [p for p in polytopes if len(p)>1]
    #print("Final polytopes:", polytopes)
    return polytopes

In [7]:
model_results_dict = {}
cf_results_dict = {}
cm_results_dict = {}
cfs_good_indices = {}
for c1, dataset_in in enumerate(datasets):
    print("Working on dataset ", str(c1))
    X = dataset_in[0]
    y = dataset_in[1]
    df = pd.DataFrame(X.values, columns=[str(i) for i in range(n_features)])
    df_train, df_test, y_train, y_test, indices_train, indices_test = train_test_split(df, y, np.arange(len(X)), test_size=0.33, random_state=42)
    
    cat_feat2 = [str(i) for i in range(len(cat_feat))]
    enc = OneHotEncoder(drop='first', min_frequency=None, sparse_output=False)
    enc.fit(df_train[cat_feat2])
    X_enc = pd.DataFrame(np.hstack([np.ones((len(df_train),1)), enc.transform(df_train[cat_feat2])]))
    polytopes = get_polytopes(enc)
    
    # # ommitted variables change
    # df_train = df_train.drop(['4','8','9'], axis=1).rename({'5':'4','6':'5','7':'6'},axis=1)
    # df_test = df_test.drop(['4','8','9'], axis=1).rename({'5':'4','6':'5','7':'6'},axis=1)
    # num_feat = [4,5,6]
    # cat_feat = [0,1,2,3]
    
    # if non_iid:
    #     outlier_indices = np.random.choice(len(df_test), n_samples//10, replace=False)
    #     df_test.iloc[outlier_indices, 5:] *= 5
    
    cf_calc = CFCalculations(
        df_train,
        y_train, 
        df_test.iloc[0:3000], # for speed
        y_test[0:3000],
        [int(i) for i in range(len(cat_feat))],
        [int(i) for i in range(len(cat_feat),len(num_feat)+len(cat_feat))],
        polytopes, 
        do_marg=False,
        min_frequency=None) # no constant assummed
        
    print(pd.DataFrame([(model, metric, value) for (model, metric), value in cf_calc.model_stats.items()], 
                       columns=['Model', 'Metric', 'Value']))
    model_results_dict[f"noise_{c1:.0f}"] = cf_calc.model_stats
    cm_results_dict[f"noise_{c1:.0f}"] = cf_calc.cms
    cfs = cf_calc.get_cfs()
    cfs_good_indices[f"noise_{c1:.0f}"] = cf_calc.check_cfs()
    cf_results_dict[f"noise_{c1:.0f}"] = cfs

    dump(cf_calc, "cf_calc_gsc.pkl")
    dump(cf_results_dict, "cf_results_dict_gsc.pkl")
    dump(model_results_dict, "model_results_gsc.pkl")
    dump(cfs_good_indices, "cfs_good_indices_gsc.pkl")
    # dump(datasets, "datasets_"+str(mock_data)+"_adult_income.pkl")
    print("==================================================================================================")

Working on dataset  0


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [b]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 669 seconds.
There were 6000 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


94/94 [==============================] - 0s 417us/step - loss: 0.1870 - accuracy: 0.9347 - precision: 0.4767
TF Test Accuracy: 93.47%
94/94 [==============================] - 0s 358us/step
  Model     Metric     Value
0    lr   accuracy  0.935333
1    lr  precision  0.904088
2   blr   accuracy  0.904667
3   blr  precision  0.880273
4    rf   accuracy  0.936333
5    rf  precision  0.917786
6    tf   accuracy  0.934667
7    tf  precision  0.918009
Working on lr skl
Working on lr blr_mean
Working on lr NICE
Working on lr DiCE


100%|██████████| 3000/3000 [06:59<00:00,  7.15it/s]


Working on lr RL


...

/Users/leo/anaconda3/envs/pymc5b/lib/python3.11/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/Users/leo/anaconda3/envs/pymc5b/lib/python3.11/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


CFs for nice tf model ok
94/94 [==============================] - 0s 328us/step
CFs for dice tf model ok
CFs checked
Working on dataset  6


/Users/leo/anaconda3/envs/pymc5b/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Auto-assigning NUTS sampler...
INFO:pymc.sampling.mcmc:Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
INFO:pymc.sampling.mcmc:Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
INFO:pymc.sampling.mcmc:Multiprocess sampling (4 chains in 4 jobs)
NUTS: [b]
INFO:pymc.sampling.mcmc:NUTS: [b]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 153 seconds.
INFO:pymc.sampling.mcmc:Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 153 seconds.
/Users/leo/anaconda3/envs/pymc5b/lib/python3.11/site-packages/sklearn/met

94/94 [==============================] - 0s 546us/step - loss: 0.4649 - accuracy: 0.8313 - precision_6: 0.2692
TF Test Accuracy: 83.13%
94/94 [==============================] - 0s 410us/step
  Model     Metric     Value
0    lr   accuracy  0.835333
1    lr  precision  0.697782
2   blr   accuracy  0.835333
3   blr  precision  0.697782
4    rf   accuracy  0.834667
5    rf  precision  0.739044
6    tf   accuracy  0.831333
7    tf  precision  0.742879
Working on lr skl
Working on lr blr_mean
Working on lr NICE


ValueError: attempt to get argmin of an empty sequence

In [10]:
dataset_in[0]

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
71842,-748.057136,68.242502,0.469601,8.625553,61516.996004,9.439866,-0.672495,5.362390,2.927628,2.088054
12938,-1074.525176,122.957738,5.347511,1139.475192,-37074.820146,26.406422,-8.445003,3.766982,0.873497,0.400499
69944,-927.286233,94.551447,-0.226430,-1450.239939,49809.567166,5.757193,1.700784,2.827809,5.616674,3.082127
121097,-610.689979,47.451570,16.690549,1442.144160,29622.413384,43.885317,-2.083267,2.637165,-2.666276,4.482875
123388,-690.639153,8.691644,1.504320,2200.703008,24005.882714,28.305026,-5.947135,1.637502,10.472807,4.873774
...,...,...,...,...,...,...,...,...,...,...
59909,790.162295,43.871572,-0.981665,-934.933262,97773.082598,21.339958,-9.488686,5.087998,-0.874774,2.904520
60843,-452.459696,16.532955,2.235577,1175.312677,-23224.488752,-0.393805,4.288420,-1.947462,-3.314082,2.920196
124535,-796.063940,80.161130,-16.275576,-654.325257,106108.174150,20.397587,0.410121,1.187618,6.082625,-2.463208
44257,-1119.143264,60.069993,1.253826,1003.535073,-27087.127855,-13.823526,-8.801263,3.109969,-3.598563,-2.318279
